# Практика: Метод опорных векторов (SVM)

## Что вы сделаете

В этом ноутбуке вы:

1. Загрузите датасет Breast Cancer и проведёте базовый EDA;
2. Обучите **LinearSVC** — быстрый линейный SVM;
3. Исследуете влияние **масштабирования признаков** на качество;
4. Подберёте гиперпараметры (**C**, **kernel**, **gamma**) через `GridSearchCV`;
5. Визуализируете **границу решений** на двух признаках;
6. Проанализируете **ошибки** модели и опорные векторы;
7. Сравните линейный и RBF SVM.

**Датасет:** [Breast Cancer Wisconsin](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html) — бинарная классификация (злокачественная / доброкачественная опухоль), 569 примеров, 30 числовых признаков.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print('Импорты выполнены успешно!')

In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

print('Размер датасета:', X.shape)
print('Классы:', data.target_names)
print('Распределение классов:')
print(y.value_counts())

display(X.head())

In [ ]:
print('Пропуски по признакам:')
display(X.isna().sum().sum())

print('\nБазовая статистика первых 5 признаков:')
display(X.iloc[:, :5].describe().T.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
features_to_plot = ['mean radius', 'mean texture', 'mean area']

for ax, feat in zip(axes, features_to_plot):
    for cls, label, color in [(0, 'malignant', 'tomato'), (1, 'benign', 'steelblue')]:
        ax.hist(X.loc[y == cls, feat], bins=30, alpha=0.6, label=label, color=color)
    ax.set_title(feat)
    ax.legend()

plt.suptitle('Распределение признаков по классам', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Баланс классов в train:', y_train.value_counts().to_dict())
print('Баланс классов в test:', y_test.value_counts().to_dict())

In [ ]:
svm_no_scale = SVC(kernel='rbf', C=1.0, random_state=42)
svm_no_scale.fit(X_train, y_train)
y_pred_no_scale = svm_no_scale.predict(X_test)

print('Accuracy без масштабирования:', accuracy_score(y_test, y_pred_no_scale))
print('F1 без масштабирования:', f1_score(y_test, y_pred_no_scale))

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=1.0, random_state=42))
])

pipe.fit(X_train, y_train)
y_pred_with_scale = pipe.predict(X_test)

print('Accuracy с масштабированием:', accuracy_score(y_test, y_pred_with_scale))
print('F1 с масштабированием:', f1_score(y_test, y_pred_with_scale))
print('\nКлассификационный отчёт:')
print(classification_report(y_test, y_pred_with_scale, target_names=data.target_names))

In [ ]:
param_grid = {
    'svm__C': [0.01, 0.1, 1, 10, 100],
    'svm__kernel': ['linear', 'rbf'],
    'svm__gamma': ['scale', 0.01, 0.1],
}

base_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(random_state=42))
])

search = GridSearchCV(base_pipe, param_grid, cv=5, scoring='f1', n_jobs=-1)
search.fit(X_train, y_train)

print('Лучшие параметры:', search.best_params_)
print('CV F1 (best):', search.best_score_)

In [ ]:
best_model = search.best_estimator_
y_pred_best = best_model.predict(X_test)

print('Test accuracy (best):', accuracy_score(y_test, y_pred_best))
print('Test F1 (best):', f1_score(y_test, y_pred_best))
print(classification_report(y_test, y_pred_best, target_names=data.target_names))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best,
    display_labels=data.target_names,
    ax=ax,
    colorbar=False,
    cmap='Blues'
)
ax.set_title('Confusion Matrix — лучшая модель')
plt.tight_layout()
plt.show()

In [ ]:
def plot_decision_boundary(model, X_2d, y, title='Граница решений SVM'):
    h = 0.05
    x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
    y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                          np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, alpha=0.25,
                 cmap=mcolors.ListedColormap(['tomato', 'steelblue']))
    plt.contour(xx, yy, Z, colors='k', linewidths=0.8)
    colors = ['tomato' if c == 0 else 'steelblue' for c in y]
    plt.scatter(X_2d[:, 0], X_2d[:, 1], c=colors, edgecolors='k', s=40, alpha=0.8)
    plt.xlabel('mean radius (scaled)')
    plt.ylabel('mean texture (scaled)')
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
feat1, feat2 = 'mean radius', 'mean texture'

X_train_2d = X_train[[feat1, feat2]].values
X_test_2d = X_test[[feat1, feat2]].values

best_C = search.best_params_['svm__C']
best_gamma = search.best_params_['svm__gamma']

pipe_2d = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=best_C, gamma=best_gamma, random_state=42))
])

pipe_2d.fit(X_train_2d, y_train)

plot_decision_boundary(pipe_2d, X_train_2d, y_train.values,
                       title='Train: Граница решений (2 признака)')
plot_decision_boundary(pipe_2d, X_test_2d, y_test.values,
                       title='Test: Граница решений (2 признака)')

In [ ]:
errors_idx = np.where(y_pred_best != y_test)[0]

print(f'Число ошибочных предсказаний: {len(errors_idx)}')

errors_df = X_test.iloc[errors_idx, :5].copy()
errors_df['true_label'] = y_test.values[errors_idx]
errors_df['pred_label'] = y_pred_best[errors_idx]
errors_df['true_class'] = errors_df['true_label'].map({0: 'malignant', 1: 'benign'})
errors_df['pred_class'] = errors_df['pred_label'].map({0: 'malignant', 1: 'benign'})

display(errors_df)

### Анализ ошибок

**Типы ошибок:**
- False Positive (FP): предсказали benign, на самом деле malignant
- False Negative (FN): предсказали malignant, на самом деле benign

**В медицинской задаче более критичны FN** (пропустить злокачественную опухоль), чем FP (лишние обследования).

In [ ]:
pipe_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', LinearSVC(C=best_C, max_iter=5000, random_state=42))
])

pipe_rbf = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=best_C, gamma=best_gamma, random_state=42))
])

for name, model in [('LinearSVC', pipe_linear), ('RBF SVC', pipe_rbf)]:
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1')
    print(f'{name}: F1 = {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
def get_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
    }

results = pd.DataFrame([
    get_metrics(y_test, y_pred_no_scale),
    get_metrics(y_test, y_pred_with_scale),
    get_metrics(y_test, y_pred_best),
], index=['no_scale', 'with_scale', 'best_grid'])

display(results.round(4))

## Ответы на вопросы

**1. Почему SVM необходимо масштабировать признаки?**
SVM использует евклидово расстояние для margin и ядер. Разные масштабы признаков искажают геометрию.

**2. Что произойдёт с границей решений при изменении C?**
- Увеличение C → меньше ошибок на train, сложная граница, переобучение
- Уменьшение C → шире margin, гладкая граница, лучше обобщение

**3. Что такое опорные векторы?**
Обучающие примеры на границе margin. Только они определяют гиперплоскость.

**4. Когда выбрать линейное ядро вместо RBF?**
При большом количестве признаков, линейной разделимости, или когда нужна интерпретируемость.

**5. Почему нельзя обучать StandardScaler на всей выборке перед CV?**
Data leakage — информация из теста попадает в train. Scaler нужно обучать только на train внутри каждого fold.